# SEN2SR Lite: 10-Band 3-Way Spectral & Spatial Comparison
### Comparing: Normal Captured Input vs. Pretrained Baseline vs. Fine-Tuned SEN2SR Lite

This notebook evaluates your Sentinel-2 study scene (`Browser_images_1.zip` / `Browser_images.zip`) across all **10 multispectral bands** (`B02, B03, B04, B05, B06, B07, B08, B8A, B11, B12`).

**The Three Competing Representations:**
1. **Actual Normal Captured Input:** Raw Sentinel-2 (10m) surface reflectance upscaled 4× via bilinear interpolation.
2. **Pretrained Baseline SEN2SR Lite:** Off-the-shelf Taco Foundation generator.
3. **Fine-Tuned SEN2SR Lite (Compound Loss):** Model powered by your fine-tuned checkpoint (`sen2sr_lite_compound_finetuned.pth`).

**Comprehensive Evaluation Layers:**
- **Overall Multispectral:** Average reflectance across all 10 bands.
- **RGB True Color:** (B04, B03, B02) natural composite.
- **NDVI (Vegetation Index):** $\frac{\text{B08} - \text{B04}}{\text{B08} + \text{B04}}$
- **NDWI (Water/Moisture Index):** $\frac{\text{B03} - \text{B08}}{\text{B03} + \text{B08}}$
- **NDBI (Built-Up / Soil Index):** $\frac{\text{B11} - \text{B08}}{\text{B11} + \text{B08}}$
- **Difference Heatmaps & Diagnostic Metrics Table:** Quantifying spectral stability and edge sharpening.

## Cell 1 — Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, glob, time, warnings, logging, zipfile, shutil, re

# Strict suppression of GDAL TIFF warnings
os.environ['CPL_LOG_ERRORS'] = 'OFF'
os.environ['CPL_LOG'] = 'OFF'
os.environ['GDAL_PAM_ENABLED'] = 'NO'
warnings.filterwarnings('ignore')
for logger_name in ['rasterio', 'rasterio._env', 'rasterio.env', 'rasterio._base']:
    l = logging.getLogger(logger_name)
    l.setLevel(logging.ERROR)
    l.propagate = False

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

BASE_DIR = '/content/drive/MyDrive/Project_Folder'
DATASETS_DIR = os.path.join(BASE_DIR, 'Datasets')
OUTPUT_DIR = os.path.join(BASE_DIR, 'Results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, 'sen2sr_lite_compound_finetuned.pth')
print(f'Fine-tuned checkpoint exists: {os.path.exists(CHECKPOINT_PATH)}')
if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f'Checkpoint not found at: {CHECKPOINT_PATH}')

## Cell 2 — Install Required Libraries

In [ ]:
%pip install -q sen2sr mlstac rasterio safetensors pandas matplotlib

## Cell 3 — Extract 10-Band ZIP from Google Drive

Automatically searches for `Browser_images_1.zip` or `Browser_images.zip` in `Datasets/` and extracts the 10 raw band TIFFs.

In [ ]:
# ---------------------------------------------------------------------------
# Specify your ZIP filename here (e.g. 'Browser_images_2.zip', 'Browser_images_1.zip', etc.)
# ---------------------------------------------------------------------------
INPUT_ZIP_NAME = 'Browser_images_2.zip'

ZIP_PATH = None
target_path = os.path.join(DATASETS_DIR, INPUT_ZIP_NAME)

if os.path.exists(target_path):
    ZIP_PATH = target_path
elif os.path.exists(f'/content/{INPUT_ZIP_NAME}'):
    ZIP_PATH = f'/content/{INPUT_ZIP_NAME}'
else:
    # Auto-fallback: search for any Browser_images*.zip in Datasets or /content
    found_zips = sorted(glob.glob(os.path.join(DATASETS_DIR, 'Browser_images*.zip')) + glob.glob('/content/Browser_images*.zip'))
    if found_zips:
        ZIP_PATH = found_zips[-1] # automatically picks the latest ZIP

if not ZIP_PATH:
    raise FileNotFoundError(f'Could not find {INPUT_ZIP_NAME} or any Browser_images*.zip in {DATASETS_DIR}')

print(f'Using input ZIP archive: {ZIP_PATH}')
EXTRACT_DIR = '/content/copernicus_data'
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    archive.extractall(EXTRACT_DIR)

raster_files = []
for root, _, files in os.walk(EXTRACT_DIR):
    for f in files:
        if f.lower().endswith(('.tif', '.tiff', '.jp2')):
            raster_files.append(os.path.join(root, f))

print(f'Extracted {len(raster_files)} raster files.')

## Cell 4 — Align and Stack All 10 Sentinel-2 Bands

Stack order: `B02, B03, B04, B05, B06, B07, B08, B8A, B11, B12` (reprojected to the reference 10m grid).

In [ ]:
import rasterio
from rasterio.warp import reproject, Resampling

BAND_NAMES = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']

# Identify individual band files
found_bands = {}
for path in raster_files:
    upper = os.path.basename(path).upper()
    matches = [b for b in BAND_NAMES if re.search(rf'(^|[_.\-]){b}([_.\-]|$)', upper)]
    if matches:
        found_bands[matches[0]] = path

missing = [b for b in BAND_NAMES if b not in found_bands]
if missing:
    raise ValueError(f'Missing required bands: {missing}')

print('All 10 Sentinel-2 bands successfully matched:')
for b in BAND_NAMES:
    print(f' - {b}: {os.path.basename(found_bands[b])}')

# Reference grid is B02 (10m resolution)
with rasterio.open(found_bands['B02']) as ref:
    ref_profile = ref.profile.copy()
    ref_transform, ref_crs = ref.transform, ref.crs
    height, width = ref.height, ref.width

stack = []
for b in BAND_NAMES:
    with rasterio.open(found_bands[b]) as src:
        dest = np.zeros((height, width), dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dest,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.bilinear
        )
        stack.append(dest)

raw_stack = np.stack(stack).astype(np.float32)

# Scale reflectance to physical [0.0, 1.0]
finite_vals = raw_stack[np.isfinite(raw_stack)]
max_val = np.percentile(finite_vals, 99)
scale_factor = 10000.0 if max_val > 255.0 else (255.0 if max_val > 2.0 else 1.0)
X_np = np.nan_to_num(raw_stack / scale_factor, nan=0.0, posinf=1.0, neginf=0.0)
X_10band = torch.from_numpy(X_np).float().to(device)

print(f'\n10-Band Input Tensor Shape: {tuple(X_10band.shape)}')
print(f'Reflectance Range (1st-99th percentile): {np.percentile(X_np, 1):.4f} to {np.percentile(X_np, 99):.4f}')

## Cell 5 — Load Both Pretrained Baseline & Fine-Tuned 10-Band Generators

1. **Baseline Model:** Pretrained 10-band compiled model downloaded from Taco Foundation.
2. **Fine-Tuned Model:** Compiled model with Stage 1 RGBN generator replaced by your fine-tuned weights (`sen2sr_lite_compound_finetuned.pth`).

In [ ]:
import mlstac
from sen2sr.models.opensr_baseline.cnn import CNNSR

MODEL_DIR = '/content/model/SEN2SRLite'
os.makedirs(MODEL_DIR, exist_ok=True)
mlstac.download(
    file='https://huggingface.co/tacofoundation/SEN2SR/resolve/main/SEN2SRLite/main/mlm.json',
    output_dir=MODEL_DIR
)

# 1. Load Pretrained Baseline 10-band model
model_baseline = mlstac.load(MODEL_DIR).compiled_model(device=device)
model_baseline = model_baseline.to(device).eval()
print('1. Pretrained Baseline 10-Band SEN2SR Lite loaded successfully.')

# 2. Load Fine-Tuned 10-band model and replace Stage 1 weights with compound fine-tuned checkpoint
model_finetuned = mlstac.load(MODEL_DIR).compiled_model(device=device)
weights_finetuned = torch.load(CHECKPOINT_PATH, map_location=device)

# SEN2SR Lite compiled_model hierarchy:
# model_finetuned (SRModel from referencex4) -> sr_model (SRModelWithConstraint from nonreference) -> sr_model (Stage 1 CNNSR)
injected = False
if hasattr(model_finetuned, 'sr_model') and hasattr(model_finetuned.sr_model, 'sr_model'):
    try:
        model_finetuned.sr_model.sr_model.load_state_dict(weights_finetuned)
        injected = True
        print('   - Injected fine-tuned weights directly into model.sr_model.sr_model (CNNSR)!')
    except Exception as e:
        print(f'   - Direct hierarchy load failed: {e}. Trying submodule search...')

if not injected:
    for name, m in model_finetuned.named_modules():
        if hasattr(m, 'conv_1') and hasattr(m, 'conv_2'):
            try:
                m.load_state_dict(weights_finetuned)
                injected = True
                print(f"   - Injected fine-tuned weights into submodule '{name}' ({type(m).__name__})!")
                break
            except Exception:
                continue

assert injected, 'CRITICAL ERROR: Failed to inject fine-tuned weights into SEN2SR Lite Stage 1 CNNSR!'

# 3. Mathematically verify that Fine-Tuned weights differ from Pretrained Baseline
p_base = next(model_baseline.sr_model.sr_model.parameters())
p_fine = next(model_finetuned.sr_model.sr_model.parameters())
weight_diff_norm = torch.norm(p_fine - p_base).item()
print(f'Weight Difference Norm (Fine-Tuned vs Baseline): {weight_diff_norm:.6f}')
assert weight_diff_norm > 1e-4, f'CRITICAL: Weights did not update (diff={weight_diff_norm})! Models are identical.'

model_finetuned = model_finetuned.to(device).eval()
print('2. Fine-Tuned 10-Band SEN2SR Lite (Compound Loss) loaded & verified successfully!')

## Cell 6 — Execute 3-Way Inference across the Entire Scene

Inference uses seamless overlapping sliding-window patching to ensure zero memory exhaustion on any scene size.

In [ ]:
def run_sliding_window(model, X, patch_size=128, stride=48, scale=4):
    ps_hr = patch_size * scale
    _, H, W = X.shape
    pad_h = max(0, patch_size - H)
    pad_w = max(0, patch_size - W)
    X_pad = F.pad(X, (0, pad_w, 0, pad_h), mode='replicate')
    Hp, Wp = X_pad.shape[-2:]
    
    # Smooth 2D Hann Cosine window: eliminates patch seam boundaries & urban jitter
    wy = torch.hann_window(ps_hr, periodic=False, device=device)
    wx = torch.hann_window(ps_hr, periodic=False, device=device)
    window = torch.outer(wy, wx).unsqueeze(0).clamp(min=1e-3)
    
    out = torch.zeros((10, Hp * scale, Wp * scale), device=device, dtype=X.dtype)
    weights = torch.zeros((1, Hp * scale, Wp * scale), device=device, dtype=X.dtype)
    
    y_steps = list(range(0, Hp - patch_size + 1, stride))
    if (Hp - patch_size) not in y_steps:
        y_steps.append(Hp - patch_size)
    x_steps = list(range(0, Wp - patch_size + 1, stride))
    if (Wp - patch_size) not in x_steps:
        x_steps.append(Wp - patch_size)
        
    with torch.inference_mode():
        for y0 in y_steps:
            for x0 in x_steps:
                patch = X_pad[:, y0:y0+patch_size, x0:x0+patch_size].unsqueeze(0)
                pred = model(patch).squeeze(0)
                oy, ox = y0 * scale, x0 * scale
                out[:, oy:oy+ps_hr, ox:ox+ps_hr] += pred * window
                weights[:, oy:oy+ps_hr, ox:ox+ps_hr] += window
                
    super_res = out / weights.clamp_min(1e-6)
    return super_res[:, :H*scale, :W*scale]

print('Running 3-Way Inference...')
t0 = time.time()

# 1. Actual normal captured input upscaled 4x (Bilinear 10m)
X_upscaled = F.interpolate(X_10band.unsqueeze(0), scale_factor=4, mode='bilinear', align_corners=False).squeeze(0)

# 2. Pretrained Baseline SEN2SR Lite
superX_baseline = run_sliding_window(model_baseline, X_10band)

# 3. Fine-Tuned SEN2SR Lite (Compound Loss)
superX_finetuned = run_sliding_window(model_finetuned, X_10band)

elapsed = time.time() - t0
print(f'All 3 models executed in {elapsed:.2f} seconds!')
print(f'Input (4x upscaled) shape: {tuple(X_upscaled.shape)}')
print(f'Baseline output shape:     {tuple(superX_baseline.shape)}')
print(f'Fine-tuned output shape:   {tuple(superX_finetuned.shape)}')

## Cell 7 — Calculate Overall Multispectral, RGB, and All Spectral Indices

- **Overall Multispectral:** Mean across all 10 bands.
- **RGB:** Red = B04 (idx 2), Green = B03 (idx 1), Blue = B02 (idx 0).
- **NDVI:** $\frac{\text{B08 (idx 6)} - \text{B04 (idx 2)}}{\text{B08} + \text{B04}}$
- **NDWI:** $\frac{\text{B03 (idx 1)} - \text{B08 (idx 6)}}{\text{B03} + \text{B08}}$
- **NDBI:** $\frac{\text{B11 (idx 8)} - \text{B08 (idx 6)}}{\text{B11} + \text{B08}}$

In [ ]:
def calculate_index(cube, b_a, b_b, eps=1e-6):
    a = cube[b_a]
    b = cube[b_b]
    denom = a + b
    valid = (torch.isfinite(a) & torch.isfinite(b) & (denom.abs() > eps))
    res = torch.full_like(denom, float('nan'))
    res[valid] = (a[valid] - b[valid]) / denom[valid]
    return res.cpu().numpy()

# 1. Overall Multispectral (average of all 10 bands)
overall_input = X_upscaled.mean(dim=0).cpu().numpy()
overall_base  = superX_baseline.mean(dim=0).cpu().numpy()
overall_fine  = superX_finetuned.mean(dim=0).cpu().numpy()

# 2. RGB arrays
def to_rgb_view(cube):
    rgb = cube[[2, 1, 0]].permute(1, 2, 0).cpu().numpy()
    lo, hi = np.percentile(rgb[np.isfinite(rgb)], [2, 98])
    return np.clip((rgb - lo) / max(hi - lo, 1e-8), 0.0, 1.0)

rgb_input = to_rgb_view(X_upscaled)
rgb_base  = to_rgb_view(superX_baseline)
rgb_fine  = to_rgb_view(superX_finetuned)

# 3. Spectral Indices
# Band Indices in 10-band stack: B02:0, B03:1, B04:2, B05:3, B06:4, B07:5, B08:6, B8A:7, B11:8, B12:9
B03, B04, B08, B11 = 1, 2, 6, 8

ndvi_input = calculate_index(X_upscaled, B08, B04)
ndvi_base  = calculate_index(superX_baseline, B08, B04)
ndvi_fine  = calculate_index(superX_finetuned, B08, B04)

ndwi_input = calculate_index(X_upscaled, B03, B08)
ndwi_base  = calculate_index(superX_baseline, B03, B08)
ndwi_fine  = calculate_index(superX_finetuned, B03, B08)

ndbi_input = calculate_index(X_upscaled, B11, B08)
ndbi_base  = calculate_index(superX_baseline, B11, B08)
ndbi_fine  = calculate_index(superX_finetuned, B11, B08)

print('All 5 evaluation layers computed successfully!')

## Cell 8 — Display 5×3 Matrix Comparison

Comparing **Before (Normal Captured)** vs. **Baseline SEN2SR Lite** vs. **Fine-Tuned SEN2SR Lite** side-by-side across all 5 layers.

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(18, 25))

col_titles = [
    '1. Normal Captured (10m Upscaled)',
    '2. Pretrained Baseline SEN2SR (2.5m)',
    '3. Fine-Tuned SEN2SR Lite (2.5m, Ours)'
]

# Row 0: Overall Multispectral
ov_min, ov_max = np.percentile(overall_fine, [2, 98])
for j, img in enumerate([overall_input, overall_base, overall_fine]):
    axes[0, j].imshow(img, cmap='gray', vmin=ov_min, vmax=ov_max)
    axes[0, j].set_title(f'{col_titles[j]}\nOverall Multispectral (10-Band Mean)', fontsize=12, fontweight='bold')
    axes[0, j].axis('off')

# Row 1: RGB True Color
for j, img in enumerate([rgb_input, rgb_base, rgb_fine]):
    axes[1, j].imshow(img)
    axes[1, j].set_title(f'{col_titles[j]}\nRGB Composite (B04, B03, B02)', fontsize=12, fontweight='bold')
    axes[1, j].axis('off')

# Row 2: NDVI Vegetation Map
for j, img in enumerate([ndvi_input, ndvi_base, ndvi_fine]):
    im_ndvi = axes[2, j].imshow(img, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
    axes[2, j].set_title(f'NDVI Vegetation Index\n({col_titles[j].split(".")[1].strip()})', fontsize=11)
    axes[2, j].axis('off')
fig.colorbar(im_ndvi, ax=axes[2, :], fraction=0.015, pad=0.02, orientation='horizontal', label='NDVI Value')

# Row 3: NDWI Water / Moisture Map
for j, img in enumerate([ndwi_input, ndwi_base, ndwi_fine]):
    im_ndwi = axes[3, j].imshow(img, cmap='BrBG', vmin=-0.5, vmax=0.3)
    axes[3, j].set_title(f'NDWI Moisture Index\n({col_titles[j].split(".")[1].strip()})', fontsize=11)
    axes[3, j].axis('off')
fig.colorbar(im_ndwi, ax=axes[3, :], fraction=0.015, pad=0.02, orientation='horizontal', label='NDWI Value')

# Row 4: NDBI Built-Up Index
for j, img in enumerate([ndbi_input, ndbi_base, ndbi_fine]):
    im_ndbi = axes[4, j].imshow(img, cmap='coolwarm', vmin=-0.5, vmax=0.5)
    axes[4, j].set_title(f'NDBI Built-Up Index\n({col_titles[j].split(".")[1].strip()})', fontsize=11)
    axes[4, j].axis('off')
fig.colorbar(im_ndbi, ax=axes[4, :], fraction=0.015, pad=0.02, orientation='horizontal', label='NDBI Value')

plt.tight_layout()
comp_img_path = os.path.join(OUTPUT_DIR, 'SEN2SR_10Band_3Way_Comparison.png')
plt.savefig(comp_img_path, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f'10-Band 3-Way comparison figure saved to: {comp_img_path}')

## Cell 9 — Difference & Correction Maps (Fine-Tuned vs. Baseline)

Directly visualizes the exact mathematical adjustments $\Delta = \text{Fine-Tuned} - \text{Baseline}$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

diff_specs = [
    ('NDVI Difference (Fine-Tuned − Baseline)', ndvi_fine - ndvi_base, 'coolwarm', 0.10),
    ('NDWI Difference (Fine-Tuned − Baseline)', ndwi_fine - ndwi_base, 'BrBG', 0.10),
    ('NDBI Difference (Fine-Tuned − Baseline)', ndbi_fine - ndbi_base, 'RdBu_r', 0.15)
]

for idx, (title, diff_arr, cmap_name, lim) in enumerate(diff_specs):
    im = axes[idx].imshow(diff_arr, cmap=cmap_name, vmin=-lim, vmax=lim)
    axes[idx].set_title(title, fontsize=11, fontweight='bold')
    axes[idx].axis('off')
    plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)

plt.tight_layout()
diff_save_path = os.path.join(OUTPUT_DIR, 'SEN2SR_10Band_Difference_Maps.png')
plt.savefig(diff_save_path, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Difference maps saved to: {diff_save_path}')

## Cell 10 — Quantitative Benchmark Table & Statistics

In [ ]:
def calculate_diff_metrics(arr_before, arr_base, arr_fine, name):
    diff_fb = arr_fine - arr_base
    valid = np.isfinite(arr_before) & np.isfinite(arr_base) & np.isfinite(arr_fine)
    
    b_valid = arr_before[valid]
    base_valid = arr_base[valid]
    fine_valid = arr_fine[valid]
    d_valid = diff_fb[valid]
    
    return {
        'Index / Layer': name,
        'Mean Input (10m)': round(float(b_valid.mean()), 4),
        'Mean Baseline (2.5m)': round(float(base_valid.mean()), 4),
        'Mean Fine-Tuned (2.5m)': round(float(fine_valid.mean()), 4),
        'Fine-Tuned vs Base MAE ↓': round(float(np.abs(d_valid).mean()), 5),
        'Fine-Tuned vs Base RMSE ↓': round(float(np.sqrt(np.mean(d_valid**2))), 5),
        'Max |Diff|': round(float(np.abs(d_valid).max()), 4),
        'Pixels |Diff| > 0.05 (%)': round(float(100 * np.mean(np.abs(d_valid) > 0.05)), 2)
    }

metrics_rows = [
    calculate_diff_metrics(overall_input, overall_base, overall_fine, 'Overall Multispectral'),
    calculate_diff_metrics(ndvi_input, ndvi_base, ndvi_fine, 'NDVI (Vegetation)'),
    calculate_diff_metrics(ndwi_input, ndwi_base, ndwi_fine, 'NDWI (Moisture)'),
    calculate_diff_metrics(ndbi_input, ndbi_base, ndbi_fine, 'NDBI (Built-Up)')
]

metrics_table = pd.DataFrame(metrics_rows).set_index('Index / Layer')
print('\n' + '='*75)
print('MAJOR PROJECT: 10-BAND 3-WAY QUANTITATIVE COMPARISON TABLE')
print('='*75)
display(metrics_table)

csv_path = os.path.join(OUTPUT_DIR, 'SEN2SR_10Band_Benchmark_Metrics.csv')
metrics_table.to_csv(csv_path)
print(f'\nQuantitative table saved to: {csv_path}')

## Cell 11 — Improvement Distinguishing Diagnostic & ROI Analysis

This cell specifically isolates and spotlights **where and how much the Fine-Tuned model improved over the Pretrained Baseline**:
1. **Spatial Edge Acuity (Tenengrad Sobel Index):** Measures physical boundary sharpness gain across multispectral layers.
2. **Targeted ROI Spotlight Zoom:** Side-by-side zoomed crops of the Riverbank (NDWI) and Vegetation Canopy (NDVI) with an improvement delta overlay.
3. **1D Spatial Transect Profile:** Cross-sectional slice across a boundary proving steeper, crisper step response without ringing.

In [ ]:
print('='*75)
print('MAJOR PROJECT: MODEL IMPROVEMENT DISTINGUISHING ANALYSIS')
print('='*75)

# 1. Spatial Edge Acuity Quantification (Tenengrad Sobel Metric)
def calc_edge_acuity(img_np):
    t = torch.from_numpy(img_np).float().unsqueeze(0).unsqueeze(0)
    sx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
    sy = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
    gx = F.conv2d(t, sx, padding=1)
    gy = F.conv2d(t, sy, padding=1)
    mag = torch.sqrt(gx**2 + gy**2 + 1e-8)
    return float(mag.mean().item())

acuity_rows = []
layers_to_check = [
    ('Overall Multispectral', overall_input, overall_base, overall_fine),
    ('NDVI (Vegetation)', ndvi_input, ndvi_base, ndvi_fine),
    ('NDWI (Moisture)', ndwi_input, ndwi_base, ndwi_fine),
    ('NDBI (Built-Up)', ndbi_input, ndbi_base, ndbi_fine)
]

for name, arr_in, arr_b, arr_f in layers_to_check:
    e_in = calc_edge_acuity(arr_in)
    e_base = calc_edge_acuity(arr_b)
    e_fine = calc_edge_acuity(arr_f)
    gain_vs_base = ((e_fine - e_base) / max(e_base, 1e-6)) * 100.0
    gain_vs_input = ((e_fine - e_in) / max(e_in, 1e-6)) * 100.0
    acuity_rows.append({
        'Layer / Index': name,
        'Input Edge Acuity (10m)': round(e_in, 5),
        'Baseline Edge Acuity (2.5m)': round(e_base, 5),
        'Fine-Tuned Edge Acuity (2.5m)': round(e_fine, 5),
        'Acuity Gain vs Base (%) ↑': f"{gain_vs_base:+.2f}%",
        'Acuity Gain vs Input (%) ↑': f"{gain_vs_input:+.2f}%"
    })

acuity_df = pd.DataFrame(acuity_rows).set_index('Layer / Index')
print('\n1. Quantitative Spatial Edge Acuity & Gradient Sharpness Comparison:')
display(acuity_df)

# 2. Targeted Region-of-Interest (ROI) Visual Distinguishing & Spotlight Overlay
H, W = overall_fine.shape

# ROI 1: Riverbank / Water Boundary (around middle center)
r1_y0, r1_y1 = int(H * 0.35), int(H * 0.70)
r1_x0, r1_x1 = int(W * 0.35), int(W * 0.65)

# ROI 2: Residential / Urban & Field Boundary (left side)
r2_y0, r2_y1 = int(H * 0.25), int(H * 0.60)
r2_x0, r2_x1 = int(W * 0.05), int(W * 0.35)

rois = [
    ("ROI 1: Water-Land Riverbank Delineation", r1_y0, r1_y1, r1_x0, r1_x1, ndwi_input, ndwi_base, ndwi_fine, 'BrBG', -1, 1),
    ("ROI 2: Agricultural & Vegetation Canopy Acuity", r2_y0, r2_y1, r2_x0, r2_x1, ndvi_input, ndvi_base, ndvi_fine, 'RdYlGn', -1, 1)
]

fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(3, 4, height_ratios=[1, 1, 0.8])

for row_idx, (roi_name, y0, y1, x0, x1, a_in, a_b, a_f, cmap, vmin, vmax) in enumerate(rois):
    crop_in = a_in[y0:y1, x0:x1]
    crop_base = a_b[y0:y1, x0:x1]
    crop_fine = a_f[y0:y1, x0:x1]
    crop_diff = crop_fine - crop_base
    
    ax0 = fig.add_subplot(gs[row_idx, 0])
    ax0.imshow(crop_in, cmap=cmap, vmin=vmin, vmax=vmax)
    ax0.set_title(f"{roi_name}\n1. Normal 10m Input", fontsize=10, fontweight='bold')
    ax0.axis('off')
    
    ax1 = fig.add_subplot(gs[row_idx, 1])
    ax1.imshow(crop_base, cmap=cmap, vmin=vmin, vmax=vmax)
    ax1.set_title("2. Pretrained Baseline (2.5m)", fontsize=10, fontweight='bold')
    ax1.axis('off')
    
    ax2 = fig.add_subplot(gs[row_idx, 2])
    ax2.imshow(crop_fine, cmap=cmap, vmin=vmin, vmax=vmax)
    ax2.set_title("3. Fine-Tuned (2.5m, Ours)", fontsize=10, fontweight='bold')
    ax2.axis('off')
    
    ax3 = fig.add_subplot(gs[row_idx, 3])
    lim = max(float(np.percentile(np.abs(crop_diff[np.isfinite(crop_diff)]), 98)), 0.02)
    im_d = ax3.imshow(crop_diff, cmap='coolwarm', vmin=-lim, vmax=lim)
    ax3.set_title(f"Improvement Spotlight (Ours − Base)\n±{lim:.3f} Acuity Delta", fontsize=10, fontweight='bold')
    ax3.axis('off')
    plt.colorbar(im_d, ax=ax3, fraction=0.046, pad=0.04)

# Row 2: 1D Boundary Transect Profile Plot
ax_transect = fig.add_subplot(gs[2, :])
transect_y = int(H * 0.52)
transect_in = ndwi_input[transect_y, :]
transect_base = ndwi_base[transect_y, :]
transect_fine = ndwi_fine[transect_y, :]
x_coords = np.arange(W)

ax_transect.plot(x_coords, transect_in, label='1. Normal Captured Input (10m Upscaled)', color='#9E9E9E', linestyle='--', linewidth=1.8)
ax_transect.plot(x_coords, transect_base, label='2. Pretrained Baseline SEN2SR (2.5m)', color='#1976D2', linewidth=1.5, alpha=0.85)
ax_transect.plot(x_coords, transect_fine, label='3. Fine-Tuned SEN2SR (2.5m, Compound Loss, Ours)', color='#D32F2F', linewidth=2.2)
ax_transect.set_title(f"1D Spatial Transect Profile across River-Land Boundary (Y = {transect_y} px)\nSteeper slope proves crisper boundary transition without ringing", fontsize=11, fontweight='bold')
ax_transect.set_xlabel("Horizontal Pixel Coordinate (X)", fontsize=10)
ax_transect.set_ylabel("NDWI Index Value", fontsize=10)
ax_transect.legend(loc='lower right', frameon=True, fontsize=10)
ax_transect.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
distinguish_path = os.path.join(OUTPUT_DIR, 'SEN2SR_Improvement_Distinguishing_Report.png')
plt.savefig(distinguish_path, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\nImprovement distinguishing report saved to: {distinguish_path}')

# 3. Save Acuity Table to CSV
acuity_csv_path = os.path.join(OUTPUT_DIR, 'SEN2SR_Edge_Acuity_Improvement.csv')
acuity_df.to_csv(acuity_csv_path)
print(f'Edge acuity CSV saved to: {acuity_csv_path}')